In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f"User uploaded file {fn} with length {len(uploaded[fn])} bytes")

Saving cumulative.csv to cumulative.csv
User uploaded file cumulative.csv with length 3695322 bytes


In [39]:
import pandas as pd
import numpy as np

# 1. Ingestion (Handling spaces automatically)
df = pd.read_csv('cumulative.csv', skipinitialspace=True)

# 2. Dropping Useless and Empty Columns
cols_to_drop = ['rowid', 'kepid', 'kepoi_name', 'koi_teq_err1', 'koi_teq_err2']
df = df.drop(columns=cols_to_drop)

# 3. Handling Missing Values
df['kepler_name'] = df['kepler_name'].fillna("Unknown")

error_cols = [col for col in df.columns if 'err1' in col or 'err2' in col]
df[error_cols] = df[error_cols].fillna(df[error_cols].median())

df = df.dropna(subset=['koi_disposition'])
df['koi_score'] = df['koi_score'].fillna(df['koi_score'].median())

# 4. Outlier Capping (Winsorization)
outlier_cols = ['koi_period', 'koi_prad', 'koi_insol']
for col in outlier_cols:
    cap_value = df[col].quantile(0.99)
    df[col] = np.where(df[col] > cap_value, cap_value, df[col])

# 5. Categorical Encoding
df['koi_disposition'] = df['koi_disposition'].astype('category').cat.codes
df['koi_pdisposition'] = df['koi_pdisposition'].astype('category').cat.codes
df = pd.get_dummies(df, columns=['koi_tce_delivname'], drop_first=True)

# 6. Export to Cleaned CSV
df.to_csv('cleaned_cumulative.csv', index=False)

print("Data cleaning complete. File saved as 'cleaned_cumulative.csv'.")

Data cleaning complete. File saved as 'cleaned_cumulative.csv'.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import subprocess
import os
from google.colab import userdata

# ==========================================
# Configuration: Update these variables
# ==========================================
FOLDER_PATH = "/content/drive/MyDrive/data-cleaning"      # The folder you want to push (e.g., /content/drive/MyDrive/my_project)
GITHUB_USERNAME = "arinskyyyy"  # Your GitHub username
REPO_NAME = "data-cleaning"              # The destination repository name
# This will pop up a text box in Colab asking you to type a message
COMMIT_MESSAGE = "Update files via Colab"
USER_EMAIL = "arinskyyyy@gmail.com"     # The email tied to your GitHub account
USER_NAME = "arinskyyyy"                   # Your display name
BRANCH = "main"                           # The branch you are pushing to (usually main or master)
# ==========================================

def run_command(command):
    """Helper function to run shell commands and print the output."""
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"✅ {result.stdout.strip()}")
    else:
        print(f"❌ Error: {result.stderr.strip()}")

# 1. Retrieve the token securely from Colab Secrets
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception as e:
    raise RuntimeError("Could not find GITHUB_TOKEN. Please add it to Colab Secrets (the key icon on the left).")

# Construct the secure remote URL
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

# 2. Navigate to the target folder
if not os.path.exists(FOLDER_PATH):
    raise FileNotFoundError(f"The folder {FOLDER_PATH} does not exist.")
os.chdir(FOLDER_PATH)
print(f"Changed directory to: {os.getcwd()}")

# 3. Execute Git Commands
print("\n--- Configuring Git ---")
run_command("git init")
run_command(f'git config --global user.email "{USER_EMAIL}"')
run_command(f'git config --global user.name "{USER_NAME}"')

print("\n--- Staging and Committing ---")
run_command("git add .")
# Note: If there are no changes, commit will throw an error, which is normal.
run_command(f'git commit -m "{COMMIT_MESSAGE}"')

print("\n--- Pushing to GitHub ---")
# Remove existing origin if it exists to avoid conflicts, then add the secure one
run_command("git remote remove origin")
run_command(f"git remote add origin {REPO_URL}")
run_command(f"git branch -M {BRANCH}")

# Push changes
run_command(f"git push -u origin {BRANCH}")

Changed directory to: /content/drive/MyDrive/data-cleaning

--- Configuring Git ---
✅ Reinitialized existing Git repository in /content/drive/MyDrive/data-cleaning/.git/
✅ 
✅ 

--- Staging and Committing ---
✅ 
✅ [main 2155fb2] Update files via Colab
 3 files changed, 9571 insertions(+), 12 deletions(-)
 rewrite 01_NASA_Exoplanet_Archive_(Astrophysics)/cleaning_pipeline.ipynb (91%)
 create mode 100644 01_NASA_Exoplanet_Archive_(Astrophysics)/cumulative.csv

--- Pushing to GitHub ---
✅ 
✅ 
✅ 
✅ Branch 'main' set up to track remote branch 'main' from 'origin'.


In [ ]:
%%writefile .gitignore
# Ignore Python cache
__pycache__/
*.pyc

# Ignore dataset files

# Ignore specific folders
my_private_data/

Overwriting .gitignore
